In [ ]:
# --- cell 1 · bootstrap（任务书 §0.1 末、§0.2）—— 五个 B notebook 里这一格完全相同 ---
import pathlib, sys

# (1) 向上搜索含 npnn/ 的目录 —— 不依赖 Jupyter 的启动目录
def _find_taskb_root(start):
    for p in (start, *start.parents):
        if (p / 'npnn').is_dir():
            return p
        if (p / 'task_b' / 'npnn').is_dir():      # 从项目根启动 Jupyter 的情况
            return p / 'task_b'
    return None

_here = pathlib.Path.cwd().resolve()
TASKB_ROOT = _find_taskb_root(_here)
assert TASKB_ROOT is not None, f'从 {_here} 逐级往上找不到含 npnn/ 的目录；请从 task_b/ 或项目根启动'
sys.path.insert(0, str(TASKB_ROOT))

# (2) 物理挡住「误 import A 的 src」：把项目根从 sys.path 里摘掉（任务书 §0.1.2）
_root = TASKB_ROOT.parent
sys.path[:] = [e for e in sys.path if pathlib.Path(e or '.').resolve() != _root]

# (3) 线程预算：必须在 import numpy 之前（BLAS 首次 import 时锁死线程数）
import config
config.limit_threads()

# (4) 环境留痕 —— 报告 B.1 要用：哪个解释器、哪个 numpy
import numpy as np
print('sys.executable:', sys.executable)
print('sys.prefix    :', sys.prefix)
print('numpy         :', np.__version__)
print('TASKB_ROOT    :', TASKB_ROOT)


# 11 · 用自己的 npnn 端到端训练（第 3 阶段）

对应任务书：§3.1（三级金字塔的 L3）、§3.3（性能工程）、§3.4（网络设计）、§0.5（计时预算）。

本 notebook 的产出：
- [x] cell 1（已给，五个 notebook 一字不差，不要改）：bootstrap + 环境留痕 + 线程限制
- [ ] **计时预算先做**：单层 conv 前向/反向耗时 + epoch 外推 → 据此定网络规模（§0.5）；
      这张小表要进报告，它是「设计网络前先做性能预算」的证据，也解释了你砍了什么
- [ ] 网络定义：2–4 个 conv stage（如 32→64→128），**每个 stage 后下采样**，
      GELU + SwitchNorm2d（都用你自己的实现），GAP + Linear→100（§3.4）
- [ ] 训练循环：取 batch → 前向 → CE → 反向 → Adam → 评估；每 epoch 记指标
- [ ] ★ 训练中记录 w_BN / w_IN / w_LN 随步数变化（SN 权重演化 → 报告核心图）
- [ ] 全部落盘到 `runs/<run_name>/`：`config.json` / `metrics.csv` / `curves.png` / `params.npz`
      （落盘格式对齐 A 的 runs/ 约定；notebook 13 只读这些文件画图，不需要重训）

⏳ 纯 CPU 训练很慢：**越早放出去跑越好**（任务书时间表：10-19 就要启动）。
降级预案（§时间安排末）：子集训练 / 减 epoch —— 但 L2 整网对拍必须保留。

## 训练前先过这三关

1. **数值梯度 + torch 对拍都通过了**（第 2 阶段验收），再谈训练 —— 否则是在错误的地基上盖楼
2. **单一变量**：先跑通「无 SN 的小网」建立基线，再加 SN / 加宽加深，别一次改三样
3. **同预算锚点**：和要对比的 torch 侧模型用同一份 config（架构、epoch、lr、batch）

⚠️ 与 A 抢 CPU：`OMP_NUM_THREADS` 限到 4–8（先量 1/2/4/8/16 的拐点）。A 的 dataloader 要用 CPU。
⚠️ dtype：训练用 float32；只有对拍/梯度检验用 float64 —— 模块内部别硬编码 dtype，跟着输入走。
